# Stage 6: Upsample and Scale Metrics

Upsample final outputs to original resolution and scale all metrics.

**Input**: 
- `labelsBoneLength_v2_ds/*_bone_length_v2.nii.gz`
- `labels/*.nii.gz` (original resolution reference)
- `images/*.nii.gz` (original resolution for BV/TV)
- `metrics/*_ds.json` (downsampled metrics)

**Output**: 
- `labelsBoneLength_v2/*_bone_length_v2.nii.gz` (upsampled)
- `metrics/bone_length_metrics_v1.json` (scaled)
- `metrics/bone_length_metrics_v2.json` (scaled)

**Scaling**:
- Volumes: ×8 (2³)
- Lengths/distances: ×2
- Coordinates: ×2

In [ ]:
# Configuration
TARGET_DIR = "/mnt/c/users/mwild/firebase/perios/levi_data_1.6.26"

# Must match the factor used in 00_downsample
DOWNSAMPLE_FACTOR = 2

# BV/TV threshold for recalculation
INTENSITY_THRESHOLD = 80.0

In [ ]:
import sys
import json
from pathlib import Path
import numpy as np
from scipy import ndimage

sys.path.insert(0, str(Path('.').resolve()))
from utils import load_nifti, save_nifti, save_metrics, ensure_dir

In [ ]:
# Setup directories
target = Path(TARGET_DIR)
METRICS_DIR = target / "metrics"

# Input (downsampled)
BONE_LENGTH_DS_DIR = target / "labelsBoneLength_v2_ds"

# Reference (original resolution)
LABELS_ORIG_DIR = target / "labels"
IMAGES_ORIG_DIR = target / "images"

# Output (upsampled)
BONE_LENGTH_OUTPUT_DIR = ensure_dir(target / "labelsBoneLength_v2")

print(f"Input (downsampled): {BONE_LENGTH_DS_DIR}")
print(f"Reference labels: {LABELS_ORIG_DIR}")
print(f"Reference images: {IMAGES_ORIG_DIR}")
print(f"Output (upsampled): {BONE_LENGTH_OUTPUT_DIR}")

In [ ]:
# Load downsample info for reference shapes
downsample_info_file = METRICS_DIR / "downsample_info.json"
with open(downsample_info_file, 'r') as f:
    downsample_info = json.load(f)

# Build lookup by sample name
shape_lookup = {s['sample_name']: s for s in downsample_info['samples']}
print(f"Loaded shape info for {len(shape_lookup)} samples")

In [ ]:
def upsample_to_reference(mask_ds, reference_shape):
    """Upsample a downsampled mask to match reference shape.
    
    Uses nearest-neighbor interpolation to preserve discrete class values.
    """
    zoom_factors = [
        reference_shape[i] / mask_ds.shape[i] 
        for i in range(3)
    ]
    
    # Use nearest-neighbor (order=0) to preserve discrete class values
    upsampled = ndimage.zoom(mask_ds.astype(np.float32), zoom_factors, order=0)
    upsampled = upsampled.astype(np.uint8)
    
    # Ensure exact shape match (zoom can sometimes be off by 1)
    if upsampled.shape != reference_shape:
        result = np.zeros(reference_shape, dtype=np.uint8)
        slices = tuple(slice(0, min(s1, s2)) for s1, s2 in zip(upsampled.shape, reference_shape))
        result[slices] = upsampled[slices]
        upsampled = result
    
    return upsampled


def scale_metrics(metrics, factor):
    """Scale metrics from downsampled to original resolution.
    
    - Volumes: ×factor³
    - Lengths/distances: ×factor
    - Coordinates: ×factor
    """
    scaled = metrics.copy()
    
    # Volume fields (×factor³)
    volume_fields = [
        'socket_volume', 'phalanx_volume', 'bone_volume',
        'TV_phalanx', 'BV_phalanx', 'TV_groundtruth', 'BV_groundtruth',
        'TV_primary', 'BV_primary', 'TV_secondary', 'BV_secondary'
    ]
    for field in volume_fields:
        if field in scaled and scaled[field] is not None:
            scaled[field] = int(scaled[field] * (factor ** 3))
    
    # Length/distance fields (×factor)
    length_fields = [
        'euclidean_distance', 'euclidean_distance_total',
        'bone_length_voxels', 'bone_length_euclidean',
        'socket_length_voxels', 'socket_length_euclidean',
        'line_length_total', 'line_length_inside_phalanx', 'line_length_outside_phalanx',
        'line_length_socket_segment', 'line_length_bone_segment'
    ]
    for field in length_fields:
        if field in scaled and scaled[field] is not None:
            if isinstance(scaled[field], int):
                scaled[field] = int(scaled[field] * factor)
            else:
                scaled[field] = float(scaled[field] * factor)
    
    # Coordinate fields (×factor for each component)
    coord_fields = ['socket_com', 'furthest_point', 'first_intersection']
    for field in coord_fields:
        if field in scaled and scaled[field] is not None:
            scaled[field] = [c * factor for c in scaled[field]]
    
    # Index fields (×factor)
    if 'intersection_index' in scaled and scaled['intersection_index'] is not None:
        scaled['intersection_index'] = int(scaled['intersection_index'] * factor)
    
    # BV/TV ratios remain the same (dimensionless)
    # intensity_threshold remains the same
    
    return scaled

In [ ]:
# Upsample bone length v2 visualizations
bone_length_ds_files = sorted(BONE_LENGTH_DS_DIR.glob("*_bone_length_v2.nii.gz"))
print(f"Found {len(bone_length_ds_files)} bone length files to upsample")
print("="*70)

for idx, ds_file in enumerate(bone_length_ds_files, 1):
    sample_name = ds_file.stem.replace('_bone_length_v2', '').replace('.nii', '')
    print(f"[{idx}/{len(bone_length_ds_files)}] {sample_name}")
    
    # Get reference shape
    if sample_name not in shape_lookup:
        print(f"    SKIPPED: No shape info found")
        continue
    
    ref_shape = tuple(shape_lookup[sample_name]['original_shape'])
    ref_affine = np.array(shape_lookup[sample_name]['original_affine'])
    
    try:
        # Load downsampled
        ds_data, _, ds_header = load_nifti(ds_file)
        ds_shape = ds_data.shape
        
        # Upsample
        upsampled = upsample_to_reference(ds_data, ref_shape)
        
        # Save
        output_file = BONE_LENGTH_OUTPUT_DIR / f"{sample_name}_bone_length_v2.nii.gz"
        save_nifti(upsampled, ref_affine, ds_header, output_file)
        
        print(f"    {ds_shape} -> {ref_shape}")
    except Exception as e:
        print(f"    ERROR: {e}")

In [ ]:
# Scale metrics
print("\n" + "="*70)
print("SCALING METRICS")
print("="*70)

# Load and scale v1 metrics
v1_ds_file = METRICS_DIR / "bone_length_metrics_v1_ds.json"
if v1_ds_file.exists():
    with open(v1_ds_file, 'r') as f:
        v1_metrics_ds = json.load(f)
    
    v1_metrics_scaled = [scale_metrics(m, DOWNSAMPLE_FACTOR) for m in v1_metrics_ds]
    
    v1_output_file = METRICS_DIR / "bone_length_metrics_v1.json"
    save_metrics(v1_metrics_scaled, v1_output_file)
    print(f"V1 metrics: Scaled {len(v1_metrics_scaled)} samples -> {v1_output_file}")

# Load and scale v2 metrics
v2_ds_file = METRICS_DIR / "bone_length_metrics_v2_ds.json"
if v2_ds_file.exists():
    with open(v2_ds_file, 'r') as f:
        v2_metrics_ds = json.load(f)
    
    v2_metrics_scaled = [scale_metrics(m, DOWNSAMPLE_FACTOR) for m in v2_metrics_ds]
    
    v2_output_file = METRICS_DIR / "bone_length_metrics_v2.json"
    save_metrics(v2_metrics_scaled, v2_output_file)
    print(f"V2 metrics: Scaled {len(v2_metrics_scaled)} samples -> {v2_output_file}")

In [ ]:
# Summary
print("\n" + "="*70)
print("UPSAMPLE AND SCALE COMPLETE")
print("="*70)

output_files = list(BONE_LENGTH_OUTPUT_DIR.glob("*.nii.gz"))
print(f"\nUpsampled bone length files: {len(output_files)}")
print(f"Output directory: {BONE_LENGTH_OUTPUT_DIR}")

if v2_ds_file.exists():
    print(f"\nScaled V2 metrics summary:")
    bone_lengths = [m['bone_length_euclidean'] for m in v2_metrics_scaled if m.get('bone_length_euclidean')]
    if bone_lengths:
        print(f"  Bone length: {np.mean(bone_lengths):.1f} +/- {np.std(bone_lengths):.1f} voxels")
        print(f"  Range: {min(bone_lengths):.1f} - {max(bone_lengths):.1f} voxels")

print(f"\nScaling applied:")
print(f"  Volumes: x{DOWNSAMPLE_FACTOR**3}")
print(f"  Lengths: x{DOWNSAMPLE_FACTOR}")
print(f"  Coordinates: x{DOWNSAMPLE_FACTOR}")